# 11 - Reservation Objective Sweeps

This notebook maps where strict Class 1 reservation beats pooled FCFS under objective functions from notebook 10.

The sweep compares strict reservation at different configured `Q` values against a matched FCFS baseline for the same scenario and seed.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, **kwargs):
        return iterable


def find_repo_dir(start: Path) -> Path:
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "simulation" / "engine.py").exists() and (
            candidate / "analysis" / "metrics.py"
        ).exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current notebook location.")


REPO_DIR = find_repo_dir(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from analysis.metrics import aggregate_result_row, class_result_rows
from simulation.engine import ClinicAppointmentSimulation
from simulation.model import PatientClassParams, SimulationConfig, ThresholdRule

plt.style.use("default")

## Sweep Constants

The configured `Q` values are interpreted as protected slots per day. FCFS is run with `Q = 0` and is reused as the matched baseline for every strict-reservation `Q` in the same scenario and seed.

In [ ]:
Q_VALUES = [0, 2, 4, 6, 8, 10, 12, 16, 20, 24, 28, 32]
LAMBDA_TOTAL_VALUES = [20, 32, 50, 80, 120]
CLASS_1_SHARES = [0.25, 0.50, 0.75]
SEEDS = list(range(5101, 5131))

CLASS_1_WEIGHTS = [1.0, 1.5, 2.0, 3.0]
CLASS_2_WEIGHT = 1.0
SLOT_COSTS = [0.0, 0.01, 0.02, 0.05]
WAIT_PENALTIES = [0.0, 0.02, 0.05]

UTILIZATION_FLOOR = 0.50
SERVED_RATE_FLOORS = [0.50, 0.60, 0.70]
CLASS_UTILIZATION_GAP_LIMITS = [0.10, 0.20, 0.30]

BASE_CONFIG = {
    "slots_per_day": 32,
    "horizon_days": 14,
    "burn_in_days": 30,
    "measure_days": 365,
    "cooldown_days": 14,
    "reserved_class_id": 1,
}

pd.Series(
    {
        "q_values": Q_VALUES,
        "lambda_total_values": LAMBDA_TOTAL_VALUES,
        "class_1_shares": CLASS_1_SHARES,
        "seeds": f"{SEEDS[0]}-{SEEDS[-1]}",
        "class_1_weights": CLASS_1_WEIGHTS,
        "slot_costs": SLOT_COSTS,
        "wait_penalties": WAIT_PENALTIES,
    },
    name="value",
).to_frame()

## Behavior Regimes

The broad grid varies total demand, Class 1 demand share, and whether Class 1 has better, equal, or worse behavior parameters than Class 2.

In [ ]:
BEHAVIOR_REGIMES = {
    "symmetric_baseline": {
        1: {
            "cancel_prob": 0.10,
            "balk_prob": {"threshold": 9, "low": 0.00, "high": 0.50},
            "no_show_prob": {"threshold": 6, "low": 0.00, "high": 0.30},
        },
        2: {
            "cancel_prob": 0.10,
            "balk_prob": {"threshold": 9, "low": 0.00, "high": 0.50},
            "no_show_prob": {"threshold": 6, "low": 0.00, "high": 0.30},
        },
    },
    "class_1_advantaged": {
        1: {
            "cancel_prob": 0.04,
            "balk_prob": {"threshold": 12, "low": 0.00, "high": 0.35},
            "no_show_prob": {"threshold": 9, "low": 0.00, "high": 0.18},
        },
        2: {
            "cancel_prob": 0.12,
            "balk_prob": {"threshold": 7, "low": 0.00, "high": 0.60},
            "no_show_prob": {"threshold": 5, "low": 0.05, "high": 0.38},
        },
    },
    "class_1_disadvantaged": {
        1: {
            "cancel_prob": 0.12,
            "balk_prob": {"threshold": 7, "low": 0.00, "high": 0.60},
            "no_show_prob": {"threshold": 5, "low": 0.05, "high": 0.38},
        },
        2: {
            "cancel_prob": 0.04,
            "balk_prob": {"threshold": 12, "low": 0.00, "high": 0.35},
            "no_show_prob": {"threshold": 9, "low": 0.00, "high": 0.18},
        },
    },
}


def make_scenario_grid() -> pd.DataFrame:
    rows = []
    for behavior_regime in BEHAVIOR_REGIMES:
        for lambda_total in LAMBDA_TOTAL_VALUES:
            for class_1_share in CLASS_1_SHARES:
                rows.append(
                    {
                        "scenario_id": f"{behavior_regime}|lambda={lambda_total}|share={class_1_share:.2f}",
                        "behavior_regime": behavior_regime,
                        "lambda_total": float(lambda_total),
                        "class_1_share": float(class_1_share),
                        "lambda_1": float(lambda_total * class_1_share),
                        "lambda_2": float(lambda_total * (1 - class_1_share)),
                    }
                )
    return pd.DataFrame(rows)


scenario_grid = make_scenario_grid()
scenario_grid.head()

## Simulation Helpers

In [ ]:
def safe_divide(numerator: float, denominator: float) -> float:
    return numerator / denominator if denominator else 0.0


def build_config(scenario: pd.Series, *, q: int, seed: int | None) -> SimulationConfig:
    behavior = BEHAVIOR_REGIMES[scenario["behavior_regime"]]
    classes = {}
    for class_id, lambda_value in [(1, scenario["lambda_1"]), (2, scenario["lambda_2"])]:
        params = behavior[class_id]
        classes[class_id] = PatientClassParams(
            class_id=class_id,
            lambda_per_day=float(lambda_value),
            balk_prob=ThresholdRule(**params["balk_prob"]),
            cancel_prob=float(params["cancel_prob"]),
            no_show_prob=ThresholdRule(**params["no_show_prob"]),
            value=1.0,
        )

    return SimulationConfig(
        slots_per_day=BASE_CONFIG["slots_per_day"],
        horizon_days=BASE_CONFIG["horizon_days"],
        burn_in_days=BASE_CONFIG["burn_in_days"],
        measure_days=BASE_CONFIG["measure_days"],
        cooldown_days=BASE_CONFIG["cooldown_days"],
        classes=classes,
        seed=seed,
        reserved_class_id=BASE_CONFIG["reserved_class_id"] if q > 0 else None,
        reserved_slots_per_day=int(q),
    )


def result_summary_row(result, config: SimulationConfig, fixed_values: dict) -> dict:
    aggregate = aggregate_result_row(result, fixed_values)
    class_df = pd.DataFrame(class_result_rows(result, fixed_values)).set_index("class_id")
    c1 = class_df.loc[1]
    c2 = class_df.loc[2]
    total_arrivals = aggregate["total_arrivals"]

    return {
        **fixed_values,
        "slots_per_day": config.slots_per_day,
        "horizon_days": config.horizon_days,
        "total_arrivals": total_arrivals,
        "total_served": aggregate["total_served"],
        "total_offered": aggregate["total_offered"],
        "served_rate": safe_divide(aggregate["total_served"], total_arrivals),
        "average_utilization": aggregate["average_utilization"],
        "mean_offered_booking_delay": aggregate["mean_offered_booking_delay"],
        "total_balked": aggregate["total_balked"],
        "total_no_offer": aggregate["total_no_offer"],
        "total_canceled": aggregate["total_canceled"],
        "total_no_show": aggregate["total_no_show"],
        "total_unresolved_booked": aggregate["total_unresolved_booked"],
        "class_1_arrivals": c1["arrivals"],
        "class_2_arrivals": c2["arrivals"],
        "class_1_served": c1["served"],
        "class_2_served": c2["served"],
        "class_1_offered": c1["offered"],
        "class_2_offered": c2["offered"],
        "class_1_served_rate": c1["percent_serviced"],
        "class_2_served_rate": c2["percent_serviced"],
        "min_class_served_rate": min(c1["percent_serviced"], c2["percent_serviced"]),
        "class_1_slot_utilization": c1["slot_utilization"],
        "class_2_slot_utilization": c2["slot_utilization"],
        "class_utilization_gap": abs(c1["slot_utilization"] - c2["slot_utilization"]),
        "class_1_total_offered_delay": c1["total_offered_booking_delay"],
        "class_2_total_offered_delay": c2["total_offered_booking_delay"],
    }


def validate_accounting(run_df: pd.DataFrame, tol: float = 1e-9) -> None:
    partition = (
        run_df["total_served"]
        + run_df["total_balked"]
        + run_df["total_no_offer"]
        + run_df["total_canceled"]
        + run_df["total_no_show"]
        + run_df["total_unresolved_booked"]
    )
    if (partition - run_df["total_arrivals"]).abs().max() > tol:
        raise AssertionError("Outcomes do not partition arrivals.")


def run_policy_grid(scenarios: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, scenario in tqdm(list(scenarios.iterrows()), desc="scenario"):
        scenario_metadata = scenario.to_dict()

        for seed in SEEDS:
            fcfs_config = build_config(scenario, q=0, seed=int(seed))
            fcfs_result = ClinicAppointmentSimulation(fcfs_config).run()
            fixed = {**scenario_metadata, "policy": "Pooled FCFS", "seed": int(seed), "Q": 0}
            rows.append(result_summary_row(fcfs_result, fcfs_config, fixed))

            for q in Q_VALUES:
                strict_config = build_config(scenario, q=int(q), seed=int(seed))
                strict_result = ClinicAppointmentSimulation(strict_config).run()
                fixed = {**scenario_metadata, "policy": "Strict C1 reservation", "seed": int(seed), "Q": int(q)}
                rows.append(result_summary_row(strict_result, strict_config, fixed))

    run_df = pd.DataFrame(rows)
    validate_accounting(run_df)
    return run_df

## Objective Helpers

In [ ]:
def weighted_mean_offered_delay(row: pd.Series, class_1_weight: float, class_2_weight: float = CLASS_2_WEIGHT) -> float:
    weighted_delay = (
        class_1_weight * row["class_1_total_offered_delay"]
        + class_2_weight * row["class_2_total_offered_delay"]
    )
    weighted_offered = (
        class_1_weight * row["class_1_offered"]
        + class_2_weight * row["class_2_offered"]
    )
    return safe_divide(weighted_delay, weighted_offered)


def objective_rows_for_run(row: pd.Series) -> list[dict]:
    rows = []
    q_share = safe_divide(row["Q"], row["slots_per_day"])

    rows.append({**row.to_dict(), "objective_name": "served_rate", "class_1_weight": 1.0, "slot_cost": 0.0, "wait_penalty": 0.0, "objective_value": row["served_rate"]})

    for weight in CLASS_1_WEIGHTS:
        weighted_served_rate = safe_divide(
            weight * row["class_1_served"] + CLASS_2_WEIGHT * row["class_2_served"],
            weight * row["class_1_arrivals"] + CLASS_2_WEIGHT * row["class_2_arrivals"],
        )
        weighted_slot_utility = weight * row["class_1_slot_utilization"] + CLASS_2_WEIGHT * row["class_2_slot_utilization"]
        weighted_delay = weighted_mean_offered_delay(row, weight)

        rows.append({**row.to_dict(), "objective_name": "priority_weighted_served_rate", "class_1_weight": weight, "slot_cost": 0.0, "wait_penalty": 0.0, "objective_value": weighted_served_rate})
        rows.append({**row.to_dict(), "objective_name": "weighted_slot_utility", "class_1_weight": weight, "slot_cost": 0.0, "wait_penalty": 0.0, "objective_value": weighted_slot_utility})

        for slot_cost in SLOT_COSTS:
            rows.append({**row.to_dict(), "objective_name": "net_priority_utility", "class_1_weight": weight, "slot_cost": slot_cost, "wait_penalty": 0.0, "objective_value": weighted_served_rate - slot_cost * q_share})

            for wait_penalty in WAIT_PENALTIES:
                delay_cost = wait_penalty * safe_divide(weighted_delay, row["horizon_days"])
                rows.append({**row.to_dict(), "objective_name": "wait_adjusted_slot_utility", "class_1_weight": weight, "slot_cost": slot_cost, "wait_penalty": wait_penalty, "objective_value": weighted_slot_utility - slot_cost * q_share - delay_cost})

    return rows


def constraint_rows_for_run(row: pd.Series) -> list[dict]:
    rows = []
    for served_floor in SERVED_RATE_FLOORS:
        for gap_limit in CLASS_UTILIZATION_GAP_LIMITS:
            feasible = (
                row["average_utilization"] >= UTILIZATION_FLOOR
                and row["min_class_served_rate"] >= served_floor
                and row["class_utilization_gap"] <= gap_limit
            )
            rows.append(
                {
                    **row.to_dict(),
                    "utilization_floor": UTILIZATION_FLOOR,
                    "served_rate_floor": served_floor,
                    "class_utilization_gap_limit": gap_limit,
                    "constraint_feasible": feasible,
                    "constrained_utilization_objective": row["average_utilization"] if feasible else np.nan,
                    "constrained_wait_objective": -row["mean_offered_booking_delay"] if feasible else np.nan,
                }
            )
    return rows


def expand_objectives(run_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    objective_df = pd.DataFrame([objective for _, row in run_df.iterrows() for objective in objective_rows_for_run(row)])
    constraint_df = pd.DataFrame([constraint for _, row in run_df.iterrows() for constraint in constraint_rows_for_run(row)])
    return objective_df, constraint_df

## Run The Sweep

This is the expensive cell. It runs 45 scenarios, 30 seeds, one FCFS baseline per scenario/seed, and strict reservation for every configured `Q`.

In [ ]:
run_df = run_policy_grid(scenario_grid)
objective_df, constraint_df = expand_objectives(run_df)

print(f"Simulation rows: {len(run_df):,}")
print(f"Objective rows: {len(objective_df):,}")
print(f"Constraint rows: {len(constraint_df):,}")

## Assertions

In [ ]:
assert (run_df.loc[run_df["policy"] == "Pooled FCFS", "Q"] == 0).all(), "FCFS must have Q = 0."

weighted_served = objective_df[objective_df["objective_name"] == "priority_weighted_served_rate"]
assert weighted_served["objective_value"].between(0, 1).all(), "Weighted served utility must be in [0, 1]."

net = objective_df[objective_df["objective_name"] == "net_priority_utility"].sort_values("slot_cost")
monotone = net.groupby(["scenario_id", "policy", "seed", "Q", "class_1_weight"])["objective_value"].apply(lambda s: s.diff().dropna().le(1e-12).all())
assert monotone.all(), "Higher slot-cost penalties should not increase net priority utility."

fcfs_by_cost = objective_df[objective_df["policy"] == "Pooled FCFS"].groupby(["scenario_id", "objective_name", "seed", "class_1_weight", "wait_penalty"])
for _, group in fcfs_by_cost:
    if group["slot_cost"].nunique() > 1:
        assert group["objective_value"].nunique() == 1, "FCFS objective should not change with slot cost because Q = 0."

validate_accounting(run_df)
print("Sweep assertions passed.")

## Pair Strict Reservation Against FCFS

In [ ]:
PAIR_KEYS = [
    "scenario_id",
    "behavior_regime",
    "lambda_total",
    "class_1_share",
    "lambda_1",
    "lambda_2",
    "seed",
    "objective_name",
    "class_1_weight",
    "slot_cost",
    "wait_penalty",
]

strict_objectives = objective_df[objective_df["policy"] == "Strict C1 reservation"].copy()
fcfs_objectives = objective_df[objective_df["policy"] == "Pooled FCFS"].copy()
fcfs_objectives = fcfs_objectives[PAIR_KEYS + ["objective_value"]].rename(columns={"objective_value": "fcfs_objective_value"})

paired_objectives = strict_objectives.merge(fcfs_objectives, on=PAIR_KEYS, how="left")
assert paired_objectives["fcfs_objective_value"].notna().all(), "Objective rows have missing FCFS matches."
paired_objectives["delta_vs_fcfs"] = paired_objectives["objective_value"] - paired_objectives["fcfs_objective_value"]

paired_objectives.head()

In [ ]:
SUMMARY_KEYS = [
    "scenario_id",
    "behavior_regime",
    "lambda_total",
    "class_1_share",
    "Q",
    "objective_name",
    "class_1_weight",
    "slot_cost",
    "wait_penalty",
]

delta_summary = (
    paired_objectives.groupby(SUMMARY_KEYS)["delta_vs_fcfs"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
delta_summary["sem"] = delta_summary["std"].fillna(0.0) / np.sqrt(delta_summary["count"])
delta_summary["ci_low"] = delta_summary["mean"] - 1.96 * delta_summary["sem"]
delta_summary["ci_high"] = delta_summary["mean"] + 1.96 * delta_summary["sem"]
delta_summary["classification"] = np.select(
    [
        (delta_summary["mean"] > 0) & (delta_summary["ci_low"] > 0),
        delta_summary["mean"] > 0,
    ],
    ["win", "candidate_win"],
    default="loss",
)

display(delta_summary.head())

## Tables

In [ ]:
best_q = (
    delta_summary.sort_values("mean", ascending=False)
    .groupby(["scenario_id", "objective_name", "class_1_weight", "slot_cost", "wait_penalty"], as_index=False)
    .head(1)
    .sort_values(["objective_name", "behavior_regime", "lambda_total", "class_1_share", "class_1_weight", "slot_cost", "wait_penalty"])
)

win_summary = (
    delta_summary.groupby(["behavior_regime", "lambda_total", "class_1_share", "objective_name", "class_1_weight", "slot_cost", "wait_penalty", "classification"])
    .size()
    .rename("num_q_values")
    .reset_index()
)

display(best_q.head(20))
display(win_summary.head(20))
display(paired_objectives[["scenario_id", "seed", "Q", "objective_name", "class_1_weight", "slot_cost", "wait_penalty", "objective_value", "fcfs_objective_value", "delta_vs_fcfs"]].head(20))

## Constraint Optimization Tables

These tables implement the meeting-note constrained views: keep filler/utilization at least 50 percent, keep class utilization gaps bounded, and require each class to have a minimum served rate.

In [ ]:
CONSTRAINT_PAIR_KEYS = [
    "scenario_id",
    "behavior_regime",
    "lambda_total",
    "class_1_share",
    "lambda_1",
    "lambda_2",
    "seed",
    "served_rate_floor",
    "class_utilization_gap_limit",
]

strict_constraints = constraint_df[constraint_df["policy"] == "Strict C1 reservation"].copy()
fcfs_constraints = constraint_df[constraint_df["policy"] == "Pooled FCFS"].copy()
fcfs_constraints = fcfs_constraints[
    CONSTRAINT_PAIR_KEYS
    + ["constraint_feasible", "constrained_utilization_objective", "constrained_wait_objective"]
].rename(
    columns={
        "constraint_feasible": "fcfs_constraint_feasible",
        "constrained_utilization_objective": "fcfs_constrained_utilization_objective",
        "constrained_wait_objective": "fcfs_constrained_wait_objective",
    }
)

paired_constraints = strict_constraints.merge(fcfs_constraints, on=CONSTRAINT_PAIR_KEYS, how="left")
paired_constraints["delta_constrained_utilization"] = paired_constraints["constrained_utilization_objective"] - paired_constraints["fcfs_constrained_utilization_objective"]
paired_constraints["delta_constrained_wait"] = paired_constraints["constrained_wait_objective"] - paired_constraints["fcfs_constrained_wait_objective"]
paired_constraints["strict_only_feasible"] = paired_constraints["constraint_feasible"] & ~paired_constraints["fcfs_constraint_feasible"]
paired_constraints["both_feasible"] = paired_constraints["constraint_feasible"] & paired_constraints["fcfs_constraint_feasible"]
paired_constraints["strict_beats_fcfs_utilization"] = paired_constraints["strict_only_feasible"] | (
    paired_constraints["both_feasible"] & (paired_constraints["delta_constrained_utilization"] > 0)
)

constraint_summary = (
    paired_constraints.groupby(["scenario_id", "behavior_regime", "lambda_total", "class_1_share", "Q", "served_rate_floor", "class_utilization_gap_limit"])
    .agg(
        strict_feasible_rate=("constraint_feasible", "mean"),
        fcfs_feasible_rate=("fcfs_constraint_feasible", "mean"),
        strict_only_feasible_rate=("strict_only_feasible", "mean"),
        strict_beats_fcfs_rate=("strict_beats_fcfs_utilization", "mean"),
        mean_delta_constrained_utilization=("delta_constrained_utilization", "mean"),
        mean_delta_constrained_wait=("delta_constrained_wait", "mean"),
    )
    .reset_index()
)

display(constraint_summary.head(20))

## Heatmap: Utility Delta By Q And Demand

In [ ]:
def plot_delta_heatmap(
    summary: pd.DataFrame,
    *,
    objective_name: str,
    behavior_regime: str = "symmetric_baseline",
    class_1_share: float = 0.50,
    class_1_weight: float = 2.0,
    slot_cost: float = 0.02,
    wait_penalty: float = 0.0,
) -> None:
    subset = summary[
        (summary["objective_name"] == objective_name)
        & (summary["behavior_regime"] == behavior_regime)
        & (summary["class_1_share"] == class_1_share)
        & (summary["class_1_weight"] == class_1_weight)
        & (summary["slot_cost"] == slot_cost)
        & (summary["wait_penalty"] == wait_penalty)
    ]
    pivot = subset.pivot(index="lambda_total", columns="Q", values="mean").sort_index()
    if pivot.empty or np.isnan(pivot.values).all():
        print("No rows for the selected heatmap filters.")
        return

    fig, ax = plt.subplots(figsize=(10, 4.5))
    image = ax.imshow(pivot.values, aspect="auto", cmap="RdBu", vmin=-np.nanmax(abs(pivot.values)), vmax=np.nanmax(abs(pivot.values)))
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([int(value) for value in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([int(value) for value in pivot.index])
    ax.set_xlabel("reserved slots per day, Q")
    ax.set_ylabel("total expected daily arrivals")
    ax.set_title(f"Delta vs FCFS: {objective_name}")
    fig.colorbar(image, ax=ax, label="strict - FCFS")
    fig.tight_layout()


plot_delta_heatmap(delta_summary, objective_name="net_priority_utility")

## Win Region Map

In [ ]:
classification_score = {"loss": 0.0, "candidate_win": 0.5, "win": 1.0}

win_region = delta_summary[
    (delta_summary["objective_name"] == "net_priority_utility")
    & (delta_summary["class_1_weight"] == 2.0)
    & (delta_summary["slot_cost"] == 0.02)
    & (delta_summary["wait_penalty"] == 0.0)
].copy()
win_region["classification_score"] = win_region["classification"].map(classification_score)
best_region = (
    win_region.sort_values(["classification_score", "mean"], ascending=False)
    .groupby(["behavior_regime", "lambda_total", "class_1_share"], as_index=False)
    .head(1)
)

if best_region.empty:
    print("No rows for the selected win-region filters.")

for behavior_regime, subset in best_region.groupby("behavior_regime"):
    pivot = subset.pivot(index="class_1_share", columns="lambda_total", values="classification_score").sort_index()
    fig, ax = plt.subplots(figsize=(7, 3.8))
    image = ax.imshow(pivot.values, aspect="auto", cmap="YlGn", vmin=0, vmax=1)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([int(value) for value in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{value:.2f}" for value in pivot.index])
    ax.set_xlabel("total expected daily arrivals")
    ax.set_ylabel("Class 1 demand share")
    ax.set_title(f"Best-Q win region: {behavior_regime}")
    fig.colorbar(image, ax=ax, label="0 loss, 0.5 candidate, 1 win")
    fig.tight_layout()

## Best-Q Curves

In [ ]:
best_q_curves = best_q[
    (best_q["objective_name"] == "net_priority_utility")
    & (best_q["behavior_regime"] == "symmetric_baseline")
    & (best_q["class_1_share"] == 0.50)
    & (best_q["slot_cost"] == 0.02)
    & (best_q["wait_penalty"] == 0.0)
]

if best_q_curves.empty:
    print("No rows for the selected best-Q filters.")
else:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for weight, group in best_q_curves.groupby("class_1_weight"):
        group = group.sort_values("lambda_total")
        ax.plot(group["lambda_total"], group["Q"], marker="o", label=f"w1={weight:g}")

    ax.set_title("Best Q by Class 1 weight")
    ax.set_xlabel("total expected daily arrivals")
    ax.set_ylabel("best reserved slots per day, Q")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()

## Slot-Cost Sensitivity

In [ ]:
sensitivity = delta_summary[
    (delta_summary["objective_name"] == "net_priority_utility")
    & (delta_summary["behavior_regime"] == "symmetric_baseline")
    & (delta_summary["class_1_share"] == 0.50)
    & (delta_summary["lambda_total"] == 50)
    & (delta_summary["class_1_weight"] == 2.0)
    & (delta_summary["wait_penalty"] == 0.0)
]

if sensitivity.empty:
    print("No rows for the selected slot-cost sensitivity filters.")
else:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for slot_cost, group in sensitivity.groupby("slot_cost"):
        group = group.sort_values("Q")
        ax.plot(group["Q"], group["mean"], marker="o", label=f"slot cost={slot_cost:g}")

    ax.axhline(0, color="0.25", linewidth=1)
    ax.set_title("Slot-cost sensitivity for strict reservation")
    ax.set_xlabel("reserved slots per day, Q")
    ax.set_ylabel("mean utility delta vs FCFS")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()